## Instalações

In [30]:
!pip install basedosdados
!pip install shapely

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 12.3 MB/s  0:00:00


## Bibliotecas

In [41]:
import basedosdados as bd
import pandas as pd
import os
from shapely import wkt
import requests

## Constantes e caminhos

In [4]:
BASE_DIR = os.getcwd()
ROOT_DIR = os.path.dirname(BASE_DIR)

# repositório dos dados em formato parquet
CAMINHO_DADOS = os.path.join(ROOT_DIR, "data")
os.makedirs(CAMINHO_DADOS, exist_ok=True)

## Consultas  base

In [25]:
consulta_1746_data_rio = '''
select * from `datario.adm_central_atendimento_1746.chamado` 
where data_particao between '2023-01-01' and '2024-12-31'
'''

consulta_bairro = '''
select * from `datario.dados_mestres.bairro` 
'''

consulta_area_planejamento = '''
select * from `datario.dados_mestres.area_planejamento` 
'''

consulta_regiao_administrativa = '''
select * from `datario.dados_mestres.regiao_administrativa` 
'''

consulta_subprefeitura = '''
select * from `datario.dados_mestres.subprefeitura` 
'''

## Funções

In [43]:
def montar_query(query, limit: int):
    """
    Objetivo:
        Adicionar obrigatoriamente uma cláusula LIMIT a uma query SQL.
    Parâmetros:
        query (str): Consulta SQL base.
        limit (int): Número máximo de registros a retornar (deve ser inteiro positivo).
    Retorno:
        str: Query final com cláusula LIMIT aplicada.
    Erros:
        TypeError: Se limit não for inteiro.
        ValueError: Se limit for menor ou igual a zero.
    """
    if not isinstance(limit, int):
        raise TypeError("O parâmetro 'limit' deve ser um inteiro.")
    if limit <= 0:
        raise ValueError("O parâmetro 'limit' deve ser maior que zero.")
    
    return f"{query}\nLIMIT {limit}"

def adicionar_centroide(df: pd.DataFrame, prefixo: str):
    """
    Objetivo:
        Calcular o centroide de geometrias no formato WKT e adicionar
        colunas de latitude e longitude ao DataFrame.
    Parâmetros:
        df (pd.DataFrame): DataFrame contendo a coluna 'geometry_wkt'.
        prefixo (str): Prefixo utilizado para nomear as colunas de saída.
    Retorno:
        pd.DataFrame: DataFrame com as colunas adicionais:
            - lat_<prefixo>
            - lon_<prefixo>
    Erros:
        TypeError: Se df não for um DataFrame ou prefixo não for string.
        ValueError: Se 'geometry_wkt' não existir ou estiver vazio.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("O parâmetro 'df' deve ser um pandas DataFrame.")
    if not isinstance(prefixo, str) or prefixo.strip() == "":
        raise TypeError("O parâmetro 'prefixo' deve ser uma string não vazia.")
    if "geometry_wkt" not in df.columns:
        raise ValueError("O DataFrame deve conter a coluna 'geometry_wkt'.")

    df = df.copy()
    # conversão para objeto geométrico
    df["geometry_obj"] = df["geometry_wkt"].apply(wkt.loads)
    # cálculo do centroide
    df[f"lon_{prefixo}"] = df["geometry_obj"].apply(lambda geom: geom.centroid.x)
    df[f"lat_{prefixo}"] = df["geometry_obj"].apply(lambda geom: geom.centroid.y)
    # remoção da coluna auxiliar
    return df.drop(columns=["geometry_obj"])

def buscar_clima_historico(latitude: float, longitude: float, data_inicio: str, data_fim: str):
    """
    Objetivo:
        Consultar a API Open-Meteo para obter dados históricos diários de clima
        (temperatura média e precipitação) para uma localização específica.
    Parâmetros:
        latitude (float): Latitude do ponto de interesse.
        longitude (float): Longitude do ponto de interesse.
        data_inicio (str): Data inicial no formato 'YYYY-MM-DD'.
        data_fim (str): Data final no formato 'YYYY-MM-DD'.
    Retorno:
        pd.DataFrame: DataFrame contendo as colunas:
            - data_particao (datetime): Data da observação
            - temperatura (float): Temperatura média diária (°C)
            - precipitacao (float): Precipitação diária acumulada (mm)
            - latitude (float): Latitude utilizada na consulta
            - longitude (float): Longitude utilizada na consulta
    Erros:
        TypeError:
            - Se latitude/longitude não forem numéricos
            - Se data_inicio/data_fim não forem strings
        ValueError:
            - Se latitude/longitude forem nulos
            - Se o intervalo de datas for inválido
        requests.exceptions.RequestException:
            - Falha na conexão com a API
        Exception:
            - Resposta inesperada da API (ex: ausência da chave 'daily')
    """
    if not isinstance(latitude, (int, float)) or not isinstance(longitude, (int, float)):
        raise TypeError("Latitude e longitude devem ser numéricos (int ou float).")

    if latitude is None or longitude is None:
        raise ValueError("Latitude e longitude não podem ser nulos.")

    if not isinstance(data_inicio, str) or not isinstance(data_fim, str):
        raise TypeError("Datas devem ser informadas como string no formato 'YYYY-MM-DD'.")

    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={latitude}"
        f"&longitude={longitude}"
        f"&start_date={data_inicio}"
        f"&end_date={data_fim}"
        "&daily=temperature_2m_mean,precipitation_sum"
        "&timezone=America%2FSao_Paulo")

    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        raise requests.exceptions.RequestException(f"Erro ao conectar na API Open-Meteo: {e}")

    dados = response.json()

    if "daily" not in dados:
        raise Exception("Resposta inválida da API: chave 'daily' não encontrada.")
    if not dados["daily"]:
        raise ValueError("Resposta da API vazia para o período informado.")


    df_clima = pd.DataFrame(dados["daily"])
    # Conversão de data
    df_clima["time"] = pd.to_datetime(df_clima["time"])
    # Padronização de nomes de colunas
    df_clima = df_clima.rename(columns={
        "time": "data_particao",
        "temperature_2m_mean": "temperatura",
        "precipitation_sum": "precipitacao"})
    # Adicionando contexto geográfico
    df_clima["latitude"] = latitude
    df_clima["longitude"] = longitude

    return df_clima   

## Gerando as bases em parquet

In [23]:
# Testando chamados 1746
df_1746_teste = bd.read_sql(montar_query(consulta_1746_data_rio, 100), billing_project_id = 'sage-sylph-494022-u2')
df_1746_teste.info(max_cols=None)
df_1746_teste.head()

Downloading: 100%|██████████|
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 34 columns):
 #   Column                            Non-Null Count  Dtype              
---  ------                            --------------  -----              
 0   id_chamado                        100 non-null    object             
 1   id_origem_ocorrencia              100 non-null    object             
 2   data_inicio                       100 non-null    datetime64[us]     
 3   data_fim                          100 non-null    datetime64[us]     
 4   id_bairro                         100 non-null    object             
 5   id_territorialidade               100 non-null    object             
 6   id_logradouro                     100 non-null    object             
 7   numero_logradouro                 98 non-null     Int64              
 8   id_unidade_organizacional         100 non-null    object             
 9   nome_unidade_organizacional       10

,id_chamado,id_origem_ocorrencia,data_inicio,data_fim,id_bairro,id_territorialidade,id_logradouro,numero_logradouro,id_unidade_organizacional,nome_unidade_organizacional,...,prazo_unidade,prazo_tipo,dentro_prazo,situacao,tipo_situacao,justificativa_status,reclamacoes,extracted_at,updated_at,data_particao
0,20324268,13,2024-05-07 13:28:16,2024-05-10 09:27:36,5,1,61374,46,166,01aGC,...,D,F,A Vencer (No Prazo),Encerrado,Atendido,None,0,2025-11-17 23:30:59.528000+00:00,2024-05-10 09:27:36,2024-05-07
1,20320361,13,2024-05-06 16:55:51,2024-05-10 09:25:43,5,1,62307,38,166,01aGC,...,D,F,A Vencer (No Prazo),Encerrado,Atendido,None,0,2025-11-17 23:30:59.528000+00:00,2024-05-10 09:25:43,2024-05-06
2,20391958,11,2024-05-22 09:37:00,2024-06-03 10:03:08,5,1,60087,37,166,01aGC,...,D,F,Em Vencimento (No Prazo),Encerrado,Atendido,None,0,2025-11-17 23:30:59.528000+00:00,2024-06-03 10:03:08,2024-05-22
3,20299547,13,2024-05-02 15:18:53,2024-05-06 15:09:36,5,1,60798,32,166,01aGC,...,D,F,A Vencer (No Prazo),Encerrado,Atendido,None,0,2025-11-17 23:30:59.528000+00:00,2024-05-06 15:09:36,2024-05-02
4,20364228,13,2024-05-15 14:19:23,2024-05-23 10:03:08,3,1,61820,75,166,01aGC,...,D,F,Em Vencimento (No Prazo),Encerrado,Atendido,None,0,2025-11-17 23:30:59.528000+00:00,2024-05-23 10:03:08,2024-05-15


In [24]:
# Criando parquet dos chamados 1746
df_1746 = bd.read_sql(consulta_1746_data_rio, billing_project_id = 'sage-sylph-494022-u2')
df_1746.to_parquet(os.path.join(CAMINHO_DADOS, "chamados_1746.parquet"), index=False)


Downloading: 100%|██████████|


In [ ]:
# Avaliando Bairro
df_bairro_teste = bd.read_sql(montar_query(consulta_bairro, 100), billing_project_id = 'sage-sylph-494022-u2')
df_bairro_teste.info(max_cols=None)
df_bairro_teste.head()

Downloading: 100%|██████████|
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id_bairro                   100 non-null    object 
 1   nome                        100 non-null    object 
 2   id_area_planejamento        100 non-null    object 
 3   id_regiao_planejamento      100 non-null    object 
 4   nome_regiao_planejamento    100 non-null    object 
 5   id_regiao_administrativa    100 non-null    object 
 6   nome_regiao_administrativa  100 non-null    object 
 7   subprefeitura               100 non-null    object 
 8   area                        100 non-null    float64
 9   perimetro                   100 non-null    float64
 10  geometry_wkt                100 non-null    object 
 11  geometry                    100 non-null    object 
dtypes: float64(2), object(10)
memory usage: 9.5+ KB


,id_bairro,nome,id_area_planejamento,id_regiao_planejamento,nome_regiao_planejamento,id_regiao_administrativa,nome_regiao_administrativa,subprefeitura,area,perimetro,geometry_wkt,geometry
0,2,Gamboa,1,1.1,Centro,1,Portuaria,Subprefeitura do Centro e Centro Histórico,1.112906e+06,4612.833630,POLYGON ((-43.18791509600112 -22.8931217219090...,"POLYGON((-43.1906945130139 -22.8921647382849, ..."
1,3,Santo Cristo,1,1.1,Centro,1,Portuaria,Subprefeitura do Centro e Centro Histórico,1.684725e+06,6743.227885,POLYGON ((-43.194498082949565 -22.903378004069...,"POLYGON((-43.194587236183 -22.9032630659357, -..."
2,4,Caju,1,1.1,Centro,1,Portuaria,Subprefeitura do Centro e Centro Histórico,5.347495e+06,19800.522524,POLYGON ((-43.22522241788446 -22.8746498473696...,MULTIPOLYGON(((-43.2252154208501 -22.874640827...
3,1,Saúde,1,1.1,Centro,1,Portuaria,Subprefeitura do Centro e Centro Histórico,3.638186e+05,2646.220568,POLYGON ((-43.1811516335027 -22.89543028498111...,"POLYGON((-43.1811873220663 -22.8954219340204, ..."
4,5,Centro,1,1.1,Centro,2,Centro,Subprefeitura do Centro e Centro Histórico,5.424754e+06,22846.244806,POLYGON ((-43.178330741495294 -22.892579047707...,MULTIPOLYGON(((-43.1783340332966 -22.892551724...


In [ ]:
# Avaliando area_planejamento (ligação por id_area_planejamento com bairro)
df_area_planejamento_teste = bd.read_sql(montar_query(consulta_area_planejamento, 100), billing_project_id = 'sage-sylph-494022-u2')
df_area_planejamento_teste.info(max_cols=None)
df_area_planejamento_teste.head()

Downloading: 100%|██████████|
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id_area_planejamento           5 non-null      object 
 1   id_area_planejamento_numerico  5 non-null      Int64  
 2   area                           5 non-null      float64
 3   perimetro                      5 non-null      float64
 4   geometry_wkt                   5 non-null      object 
 5   geometry                       5 non-null      object 
dtypes: Int64(1), float64(2), object(3)
memory usage: 377.0+ bytes


,id_area_planejamento,id_area_planejamento_numerico,area,perimetro,geometry_wkt,geometry
0,1,1,3.439537e+07,91083.782481,POLYGON ((-43.24132868014696 -22.8998070636013...,MULTIPOLYGON(((-43.2413863626158 -22.899839038...
1,2,2,1.004340e+08,100154.864528,POLYGON ((-43.28621406198833 -22.9437725812161...,MULTIPOLYGON(((-43.2871164424742 -22.943931784...
2,5,5,5.720458e+08,159745.347679,POLYGON ((-43.46929889610553 -22.8404708301563...,MULTIPOLYGON(((-43.482589818267 -22.8304275719...
3,4,4,2.937838e+08,112590.687196,POLYGON ((-43.54315356053285 -23.0386608797044...,MULTIPOLYGON(((-43.5440304135803 -23.039300246...
4,3,3,2.034919e+08,157221.830318,POLYGON ((-43.41783577453508 -22.8261681084565...,MULTIPOLYGON(((-43.4179459728856 -22.826319122...


In [ ]:
# Avaliando região administrativa (id_area_administrativa com bairro, id_area_planejamento com area_planejamento)
df_regiao_administrativa_teste = bd.read_sql(montar_query(consulta_regiao_administrativa, 100), billing_project_id = 'sage-sylph-494022-u2')
df_regiao_administrativa_teste.info(max_cols=None)
df_regiao_administrativa_teste.head()

Downloading: 100%|██████████|
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id_regiao_administrativa       33 non-null     object 
 1   nome                           33 non-null     object 
 2   id_area_planejamento           33 non-null     object 
 3   id_area_planejamento_numerico  33 non-null     Int64  
 4   id_area_planejamento_sms       33 non-null     object 
 5   area_total                     33 non-null     float64
 6   area                           33 non-null     float64
 7   perimetro                      33 non-null     float64
 8   geometry_wkt                   33 non-null     object 
 9   geometry                       33 non-null     object 
dtypes: Int64(1), float64(3), object(6)
memory usage: 2.7+ KB


,id_regiao_administrativa,nome,id_area_planejamento,id_area_planejamento_numerico,id_area_planejamento_sms,area_total,area,perimetro,geometry_wkt,geometry
0,21,Paquetá,1,1,AP 1,1.705689e+06,1.705689e+06,24841.458085,POLYGON ((-43.10571371960028 -22.7488900063676...,MULTIPOLYGON(((-43.1057206605963 -22.748866700...
1,3,Rio Comprido,1,1,AP 1,5.797238e+06,5.797238e+06,15453.190512,POLYGON ((-43.210122210625045 -22.915832359842...,"POLYGON((-43.2101925437067 -22.915828817982, -..."
2,23,Santa Teresa,1,1,AP 1,5.157142e+06,5.157142e+06,27344.839112,POLYGON ((-43.17721492507254 -22.9178789860991...,"POLYGON((-43.1776029527635 -22.9179200111748, ..."
3,7,São Cristóvão,1,1,AP 1,7.503280e+06,7.503280e+06,16336.387019,POLYGON ((-43.24132868014696 -22.8998070636013...,"POLYGON((-43.2413863626158 -22.899839038173, -..."
4,1,Portuária,1,1,AP 1,8.508944e+06,8.508944e+06,28198.682016,POLYGON ((-43.2252224178962 -22.87464984646670...,MULTIPOLYGON(((-43.2252154208501 -22.874640827...


In [ ]:
# Avaliando subprefeitura (possui chave nominal com bairro)
df_subprefeitura_teste = bd.read_sql(montar_query(consulta_subprefeitura, 100), billing_project_id = 'sage-sylph-494022-u2')
df_subprefeitura_teste.info(max_cols=None)
df_subprefeitura_teste.head()

Downloading: 100%|██████████|
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   subprefeitura  11 non-null     object 
 1   area           11 non-null     float64
 2   perimetro      11 non-null     float64
 3   geometry_wkt   11 non-null     object 
 4   geometria      11 non-null     object 
dtypes: float64(2), object(3)
memory usage: 572.0+ bytes


,subprefeitura,area,perimetro,geometry_wkt,geometria
0,Subprefeitura da Zona Oeste I,1.249448e+08,63806.440297,POLYGON ((-43.46929889610554 -22.8404708301563...,"POLYGON((-43.482589818267 -22.8304275719888, -..."
1,Subprefeitura da Zona Oeste II,1.304365e+08,67908.705364,POLYGON ((-43.51515139864343 -22.8522866130694...,"POLYGON((-43.5203942417199 -22.8465475939162, ..."
2,Subprefeitura dos Grandes Complexos,1.404889e+07,157589.060804,POLYGON ((-43.283818363298096 -22.918692912176...,MULTIPOLYGON(((-43.2838489749852 -22.918720004...
3,"Subprefeitura Barra da Tijuca, Recreio e Vargens",1.735576e+08,101584.802280,POLYGON ((-43.5113002076682 -23.06137269724783...,MULTIPOLYGON(((-43.5113499622521 -23.061365849...
4,Subprefeitura do Centro e Centro Histórico,3.265850e+07,68432.600456,POLYGON ((-43.22522241788446 -22.8746498473696...,MULTIPOLYGON(((-43.2252154208501 -22.874640827...


In [ ]:
# DFs dos auxiliares
df_bairro = bd.read_sql(consulta_bairro, billing_project_id = 'sage-sylph-494022-u2')
df_regiao_administrativa = bd.read_sql(consulta_regiao_administrativa, billing_project_id = 'sage-sylph-494022-u2')
df_area_planejamento = bd.read_sql(consulta_area_planejamento, billing_project_id = 'sage-sylph-494022-u2')
df_subprefeitura = bd.read_sql(consulta_subprefeitura, billing_project_id = 'sage-sylph-494022-u2')

Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|


In [35]:
# calculando a latitude e longitude
df_bairro = adicionar_centroide(df_bairro, "bairro")
df_regiao_administrativa = adicionar_centroide(df_regiao_administrativa, "regiao_administrativa")
df_area_planejamento = adicionar_centroide(df_area_planejamento, "area_planejamento")
df_subprefeitura = adicionar_centroide(df_subprefeitura, "subprefeitura")

In [46]:
df_bairro_dim = df_bairro[[
    "id_bairro",
    "id_regiao_administrativa",
    "id_area_planejamento",
    "nome",
    "nome_regiao_administrativa",
    "nome_regiao_planejamento",
    "subprefeitura",
    "lat_bairro",
    "lon_bairro"
]]
df_bairro_dim.rename(columns={"nome": "nome_bairro"}, inplace=True)
df_regiao_administrativa_dim = df_regiao_administrativa[[
    "id_regiao_administrativa",
    "lat_regiao_administrativa",
    "lon_regiao_administrativa"
]]
df_area_planejamento_dim = df_area_planejamento[[
    "id_area_planejamento",
    "lat_area_planejamento",
    "lon_area_planejamento"
]]
df_subprefeitura_dim = df_subprefeitura[[
    "subprefeitura",
    "lat_subprefeitura",
    "lon_subprefeitura"
]]

In [48]:
df_territorio = (
    df_bairro_dim
    .merge(df_regiao_administrativa_dim, on="id_regiao_administrativa", how="left")
    .merge(df_area_planejamento_dim, on="id_area_planejamento", how="left")
    .merge(df_subprefeitura_dim, on="subprefeitura", how="left")
)
df_territorio["id_bairro"].nunique() == len(df_territorio)
df_territorio.to_parquet(os.path.join(CAMINHO_DADOS, "dim_territorio.parquet"), index=False)